In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import sys


# Smart Plotly view of divergence summary by PRP-RA affinity-level combination
import plotly.graph_objects as go
from plotly.subplots import make_subplots

repo_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

from scripts.run_affinity_adjudication import (
    compute_pair_stats,
    normalize_abstract_index,
    affinity_level_fixed,
 )

In [ ]:

# Locate latest benchmark run
repo_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
bench_root = repo_root / 'data' / 'results' / 'affinity_benchmark'
run_dirs = sorted([p for p in bench_root.iterdir() if p.is_dir()])
if not run_dirs:
    raise FileNotFoundError(f'No benchmark runs found under {bench_root}')
run_dir = run_dirs[-1]
# Load RA-to-PRP mapping from processed CSV (generated by scripts/prepare_data.py)
mapping_path = repo_root / 'data' / 'processed' / 'ra_grouping_ra2025.csv'
ra_grouping_df = pd.read_csv(mapping_path)

required_cols = {'RA2025', 'Grouping'}
missing_cols = required_cols - set(ra_grouping_df.columns)
if missing_cols:
    raise ValueError(f'Missing required columns in mapping file: {missing_cols}')

ra_to_prp_map = (
    ra_grouping_df[['RA2025', 'Grouping']]
    .dropna(subset=['RA2025', 'Grouping'])
    .drop_duplicates()
    .rename(columns={'RA2025': 'RA2025_ID', 'Grouping': 'PRP_Name'})
)
ra_to_prp_map['RA2025_ID'] = pd.to_numeric(ra_to_prp_map['RA2025_ID'], errors='coerce').astype('Int64')

print('Mapping file:', mapping_path)
print('Mapping rows:', len(ra_to_prp_map))
display(ra_to_prp_map.head(10))

# Reading....

In [ ]:
# Locate latest benchmark run
repo_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
bench_root = repo_root / 'data' / 'results' / 'affinity_benchmark'
run_dirs = sorted([p for p in bench_root.iterdir() if p.is_dir()])
if not run_dirs:
    raise FileNotFoundError(f'No benchmark runs found under {bench_root}')
run_dir = run_dirs[-1]

ra_path_adjudication = run_dir / 'adjudicated_ra_affinities.csv'
ra_adjudication_df = pd.read_csv(ra_path_adjudication)

prp_path_adjudication = run_dir / 'adjudicated_prp_affinities.csv'
prp_adjudication_df = pd.read_csv(prp_path_adjudication)

print('Run directory:', run_dir)
print('Rows:', len(prp_adjudication_df))
display(prp_adjudication_df.head(3))


print('Run directory:', run_dir)
print('Rows:', len(ra_adjudication_df))
display(ra_adjudication_df.head(3))

## RA

In [ ]:
# Map RA questions to PRP names using data/processed/ra_grouping_ra2025.csv, then collapse abstract/PRP pairs
ra_adjudication_aggregated_df = ra_adjudication_df.copy()
ra_adjudication_aggregated_df['RA2025_ID'] = pd.to_numeric(ra_adjudication_aggregated_df['RA2025_ID'], errors='coerce').astype('Int64')
ra_adjudication_aggregated_df['Final_Affinity'] = pd.to_numeric(ra_adjudication_aggregated_df['Final_Affinity'], errors='coerce')
ra_adjudication_aggregated_df['agreement_max'] = pd.to_numeric(ra_adjudication_aggregated_df['agreement_max'], errors='coerce')

ra_adjudication_aggregated_df = ra_adjudication_aggregated_df.merge(
    ra_to_prp_map,
    on='RA2025_ID',
    how='left'
 )

ra_adjudication_aggregated_df = ra_adjudication_aggregated_df.dropna(subset=['Abstract_Index', 'PRP_Name', 'Final_Affinity']).copy()

# Keep only rows that match the max affinity inside each (Abstract_Index, PRP_Name) group
max_affinity_per_group = ra_adjudication_aggregated_df.groupby(
    ['Abstract_Index', 'PRP_Name']
 )['Final_Affinity'].transform('max')

max_rows = ra_adjudication_aggregated_df[
    ra_adjudication_aggregated_df['Final_Affinity'].eq(max_affinity_per_group)
].copy()

ra_adjudication_aggregated_df = (
    max_rows.groupby(['Abstract_Index', 'PRP_Name'], as_index=False)
    .agg(
        Final_Affinity=('Final_Affinity', 'max'),
        RA2025_ALL_ID=('RA2025_ID', lambda s: ','.join(map(str, sorted(pd.unique(s.dropna().astype(int)))))),
        agreement_max=('agreement_max', 'max')
    )
)
ra_adjudication_aggregated_df['RA2025_ID'] = ra_adjudication_aggregated_df['RA2025_ALL_ID'].apply(lambda s: s.split(',')[0] if pd.notna(s) and s else pd.NA )

ra_adjudication_aggregated_df['Affinity_Level'] = ra_adjudication_aggregated_df['Final_Affinity'].apply(affinity_level_fixed)
display(ra_adjudication_aggregated_df)

In [ ]:
ra_adjudication_aggregated_df.to_csv('ra_adjudication_aggregated.csv', index=False)


## PRP

In [ ]:
# Build analysis table and error bars
# prp_adjudication_df = prp_adjudication_df.copy()
prp_adjudication_df['Final_Affinity'] = pd.to_numeric(prp_adjudication_df['Final_Affinity'], errors='coerce')
prp_adjudication_df['Affinity_Level'] = prp_adjudication_df['Final_Affinity'].apply(affinity_level_fixed)

In [ ]:
divergence = ra_adjudication_aggregated_df[['Abstract_Index','PRP_Name', 'Final_Affinity', 'Affinity_Level','agreement_max','RA2025_ID']].copy()
divergence = divergence.merge(
    prp_adjudication_df[['Abstract_Index','PRP_Name', 'Final_Affinity', 'Affinity_Level', 'agreement_max']], on = ['Abstract_Index', 'PRP_Name'], how='inner', suffixes=('_RA', '_PRP')
)
divergence['Affinity_Divergence'] = divergence['Final_Affinity_RA'] - divergence['Final_Affinity_PRP']
divergence['Affinity_Level_Divergence_RA_PRP'] = divergence.apply(
    lambda row: f"{row['Affinity_Level_RA']} -> {row['Affinity_Level_PRP']}", axis=1
)
divergence['Coherence_Divergence'] = divergence['agreement_max_RA'] - divergence['agreement_max_PRP']
divergence.drop(columns=['Final_Affinity_RA', 'Final_Affinity_PRP', 'Affinity_Level_RA', 'Affinity_Level_PRP'], inplace=True)   

In [ ]:
# Study: Affinity_Divergence dependency by Affinity_Level_Divergence_RA_PRP
# Target: Affinity_Divergence
# Predictors (agreement): agreement_max_PRP, agreement_max_RA
analysis_cols = [
    'Affinity_Level_Divergence_RA_PRP',
    'Affinity_Divergence',
    'agreement_max_PRP',
    'agreement_max_RA',
]

group_df = divergence[analysis_cols].copy()
for c in analysis_cols[1:]:
    group_df[c] = pd.to_numeric(group_df[c], errors='coerce')
group_df = group_df.dropna(subset=['Affinity_Level_Divergence_RA_PRP', 'Affinity_Divergence'])

predictors = ['agreement_max_PRP', 'agreement_max_RA']

records = []
for aff_level_pair, g in group_df.groupby('Affinity_Level_Divergence_RA_PRP'):
    for pred in predictors:
        sub = g[['Affinity_Divergence', pred]].dropna()
        n = len(sub)

        if n >= 3 and sub[pred].nunique() > 1 and sub['Affinity_Divergence'].nunique() > 1:
            pearson_r = sub['Affinity_Divergence'].corr(sub[pred], method='pearson')
            spearman_rho = sub['Affinity_Divergence'].corr(sub[pred], method='spearman')
        else:
            pearson_r = np.nan
            spearman_rho = np.nan

        records.append({
            'Affinity_Level_Divergence_RA_PRP': aff_level_pair,
            'Predictor': pred,
            'n_pairs': n,
            'Pearson_r': pearson_r,
            'Spearman_rho': spearman_rho,
        })

dep_corr = pd.DataFrame(records)

print(f"Total rows available: {len(group_df)}")
print("Per-group correlations are computed when n_pairs >= 3 and both variables vary.")
display(dep_corr.sort_values(['Affinity_Level_Divergence_RA_PRP', 'Predictor']).reset_index(drop=True))

# Compact matrix for quick comparison (Pearson)
pearson_matrix = dep_corr.pivot(index='Affinity_Level_Divergence_RA_PRP', columns='Predictor', values='Pearson_r')
fig_heatmap = px.imshow(
    pearson_matrix,
    text_auto='.2f',
    color_continuous_scale='RdBu',
    zmin=-1,
    zmax=1,
    aspect='auto',
    title='Per Affinity-Level-Divergence Group: Pearson Correlation with Affinity_Divergence',
)
fig_heatmap.update_layout(width=950, height=500)
fig_heatmap.show()

# Grouped bars for Pearson r by predictor
plot_df = dep_corr.dropna(subset=['Pearson_r']).copy()
fig_bar = px.bar(
    plot_df,
    x='Affinity_Level_Divergence_RA_PRP',
    y='Pearson_r',
    color='Predictor',
    barmode='group',
    hover_data=['n_pairs', 'Spearman_rho'],
    title='Dependency of Affinity_Divergence with Agreement by Affinity-Level-Divergence Group',
)
fig_bar.add_hline(y=0, line_dash='dash', line_color='gray')
fig_bar.update_layout(width=1100, height=520)
fig_bar.update_xaxes(tickangle=35)
fig_bar.show()

## Dependency by Affinity-Level-Divergence Group

Study how `Affinity_Divergence` depends on `agreement_max_PRP` and `agreement_max_RA`, grouped by `Affinity_Level_Divergence_RA_PRP`.

In [ ]:
summary = (
    divergence.groupby('Affinity_Level_Divergence_RA_PRP')['Affinity_Divergence']
    .describe()
    .reset_index()
    .rename(columns={'Affinity_Level_Divergence_RA_PRP': 'Divergence_Group'})
)
summary['count_%'] = summary['count'] / summary['count'].sum() * 100
# Keep groups with more observations first for readability
summary = summary.sort_values(['count', 'mean'], ascending=[False, False]).reset_index(drop=True)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(f'Sample Size = {int(summary["count"].sum())}', 'Mean Divergence (RA - PRP)', 'Dispersion (std)'),
    horizontal_spacing=0.12
)

fig.add_trace(
    go.Bar(
        x=summary['Divergence_Group'],
        y=summary['count_%'],
        marker_color='#2A9D8F',
        name='count (%)'
    ),
    row=1, col=1
)

fig.add_trace(
    go.Bar(
        x=summary['Divergence_Group'],
        y=summary['mean'],
        error_y=dict(type='data', array=summary['std'].fillna(0), visible=True),
        marker_color='#E76F51',
        name='mean ± std'
    ),
    row=1, col=2
)

iqr = (summary['75%'] - summary['25%']).fillna(0)
# fig.add_trace(
#     go.Bar(
#         x=summary['Divergence_Group'],
#         y=summary['std'].fillna(0),
#         error_y=dict(type='data', array=iqr / 2, visible=True),
#         marker_color='#264653',
#         name='std (error ~ IQR/2)'
#     ),
#     row=1, col=3
# )

fig.add_hline(y=0, line_dash='dash', line_color='gray', row=1, col=2)

fig.update_layout(
    title='Divergence Diagnostics by Affinity-Level Pair (RA -> PRP label)',
    height=550,
    width=1600,
    showlegend=False,
    template='plotly_white',
)

# for c in [1, 2, 3]:
    # fig.update_xaxes(title_text='Affinity Level Divergence Group', tickangle=40, row=1, col=c)

fig.update_yaxes(title_text='Percentage of sample size (%)', row=1, col=1)
fig.update_yaxes(title_text='Mean divergence', row=1, col=2)
# fig.update_yaxes(title_text='Std / spread proxy', row=1, col=3)
fig.update_layout(width=700, height=500)
fig.show()

# Optional detail table for quick sorting/filtering
# display(summary[['Divergence_Group', 'count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']])

In [ ]:
summary = divergence.copy()
summary['agreement_RA'] = summary['agreement_max_RA'] == 1

summary = summary.groupby(['Affinity_Level_Divergence_RA_PRP','PRP_Name','agreement_RA'], as_index = False).agg(
    count = ('Abstract_Index','count'),
    mean = ('Affinity_Divergence','mean'),
    # agreement = ('agreement_max_PRP',),
    std = ('Affinity_Divergence','std')).rename(columns={'Affinity_Level_Divergence_RA_PRP': 'Divergence_Group'})
# count().rename(columns={'Abstract_Index': 'count'})
summary['count_%'] = summary['count'] / summary['count'].sum() * 100

fig = px.bar(
    summary,
    color='Divergence_Group',
    y='count_%',
    x='PRP_Name',
    facet_col='agreement_RA',
    barmode='stack'
)
fig.update_layout(width=700, height=500)
fig.show()

In [ ]:
# Find abstract_index with max Affinity_Divergence for each group
max_divergence_by_group = (
    divergence.loc[divergence.groupby('Affinity_Level_Divergence_RA_PRP')['Affinity_Divergence'].idxmax()]
    [['Affinity_Level_Divergence_RA_PRP', 'Abstract_Index', 'PRP_Name', 'Affinity_Divergence']]
    .sort_values('Affinity_Divergence', ascending=False)
)

print("Abstract_Index with maximum Affinity_Divergence per group:\n")
display(max_divergence_by_group)



In [ ]:
summary = divergence.copy()
summary['agreement_PRP'] = summary['agreement_max_PRP'] == 1

summary = summary.groupby(['Affinity_Level_Divergence_RA_PRP','PRP_Name','agreement_PRP'], as_index = False).agg(
    count = ('Abstract_Index','count'),
    mean = ('Affinity_Divergence','mean'),
    # agreement = ('agreement_max_PRP',),
    std = ('Affinity_Divergence','std')).rename(columns={'Affinity_Level_Divergence_RA_PRP': 'Divergence_Group'})
# count().rename(columns={'Abstract_Index': 'count'})
summary['count_%'] = summary['count'] / summary['count'].sum() * 100

fig = px.bar(
    summary,
    color='Divergence_Group',
    y='count_%',
    x='PRP_Name',
    facet_col='agreement_PRP',
    barmode='stack'
)
fig.update_layout(width=700, height=500)
fig.show()

In [ ]:
# summary = divergence.groupby(['Affinity_Level_Divergence_RA_PRP','agreement_max_PRP'], as_index = False).agg(count = ('Abstract_Index','count'), mean = ('Affinity_Divergence','mean'), std = ('Affinity_Divergence','std')).rename(columns={'Affinity_Level_Divergence_RA_PRP': 'Divergence_Group'})
# # count().rename(columns={'Abstract_Index': 'count'})
# summary['count_%'] = summary['count'] / summary['count'].sum() * 100

# fig = px.bar(
#     summary,
#     color='Divergence_Group',
#     y='count_%',
#     x='agreement_max_PRP',
#     barmode='stack'
# )
# fig.update_layout(width=700, height=500)
# fig.show()

# Plot

In [ ]:
adjudication_df = pd.concat(
    [
        prp_adjudication_df[['PRP_Name', 'Affinity_Level']].assign(source='prp'),
        ra_adjudication_aggregated_df[['PRP_Name', 'Affinity_Level']].assign(source='ra')
    ],
    ignore_index=True
)

count_df = adjudication_df.groupby(['source', 'PRP_Name', 'Affinity_Level']).size().reset_index(name='n_abstracts')
fig = px.bar(
    count_df[count_df['Affinity_Level'] != 'Low'],  # Focus on Moderate and High for clearer comparison
    x='PRP_Name',
    y='n_abstracts',
    facet_col='Affinity_Level',
    color='source',
    barmode='group',
    category_orders={
        'Affinity_Level': ['Moderate', 'High'],
        'source': ['prp', 'ra']
    },
    title='Abstract Counts by Affinity Level, PRP, and Source (PRP vs RA)'
)
fig.update_layout(
    xaxis_title='PRP Name',
    yaxis_title='Number of Abstracts',
    legend_title='Affinity Level'
)
fig.update_xaxes(tickangle=45)
fig.show()

In [ ]:
fig = px.bar(
    count_df[(count_df['Affinity_Level'] != 'Low') & (count_df['source'] != 'prp')],  # Focus on Moderate and High for clearer comparison
    x='PRP_Name',
    y='n_abstracts',
    # facet_col='Affinity_Level',
    color='Affinity_Level',
    barmode='group',
    category_orders={
        'Affinity_Level': ['Moderate', 'High'],
        'source': ['prp', 'ra']
    },
    title='Abstract Counts by RA question (highest affinity)'
)
fig.update_layout(
    xaxis_title='PRP Name',
    yaxis_title='Number of Abstracts',
    legend_title='Affinity Level'
)
# fig.update_xaxes(tickangle=45)
fig.update_layout(width=700, height=500)
fig.show()

In [ ]:
# # Deterministic counts
# prp_count_df = (
#     prp_adjudication_df.groupby(['PRP_Name', 'Affinity_Level'], as_index=False)['Abstract_Index']
#     .nunique()
#     .rename(columns={'Abstract_Index': 'n_abstracts'})
# )

# fig = px.bar(
#     prp_count_df,
#     x='PRP_Name',
#     y='n_abstracts',
#     color='Affinity_Level',
#     barmode='group',
#     category_orders={'Affinity_Level': ['Low', 'Moderate', 'High']},
#     title='Deterministic Abstract Counts by Affinity Level and PRP'
# )
# fig.update_layout(xaxis_title='PRP Name', yaxis_title='Number of Abstracts', legend_title='Affinity Level')
# fig.update_xaxes(tickangle=45)
# fig.show()
# display(prp_count_df.sort_values(['PRP_Name', 'Affinity_Level']))

In [ ]:

# ra_count_df = (
#     ra_adjudication_df.groupby(['PRP_Name', 'Affinity_Level'], as_index=False)['Abstract_Index']
#     .nunique()
#     .rename(columns={'Abstract_Index': 'n_abstracts'})
# )

# fig = px.bar(
#     ra_count_df,
#     x='PRP_Name',
#     y='n_abstracts',
#     color='Affinity_Level',
#     barmode='group',
#     category_orders={'Affinity_Level': ['Low', 'Moderate', 'High']},
#     title='RA-to-PRP Abstract Counts by Affinity Level and PRP'
# )
# fig.update_layout(xaxis_title='PRP Name', yaxis_title='Number of Abstracts', legend_title='Affinity Level')
# fig.update_xaxes(tickangle=45)
# fig.show()

# display(ra_count_df.sort_values(['PRP_Name', 'Affinity_Level']))

## PRP

In [ ]:
# Build analysis table and error bars
prp_adjudication_df = prp_adjudication_df.copy()
prp_adjudication_df['Final_Affinity'] = pd.to_numeric(prp_adjudication_df['Final_Affinity'], errors='coerce')
prp_adjudication_df['Affinity_Level'] = prp_adjudication_df['Final_Affinity'].apply(affinity_level_fixed)

## RA to PRP Aggregation and Affinity Counts

This section maps `RA2025_ID` to `PRP_Name`, collapses repeated `(Abstract_Index, PRP_Name)` rows by taking the maximum `LLM_Affinity`, and then plots deterministic affinity-level counts by PRP.


In [ ]:

# # Map RA questions to PRP names using data/processed/ra_grouping_ra2025.csv, then collapse abstract/PRP pairs
# ra_analysis_df = ra_adjudication_df.copy()
# ra_analysis_df['RA2025_ID'] = pd.to_numeric(ra_analysis_df['RA2025_ID'], errors='coerce').astype('Int64')
# ra_analysis_df['Final_Affinity'] = pd.to_numeric(ra_analysis_df['Final_Affinity'], errors='coerce')

# ra_analysis_df = ra_analysis_df.merge(
#     ra_to_prp_map,
#     on='RA2025_ID',
#     how='left'
#  )

# ra_analysis_df = ra_analysis_df.dropna(subset=['Abstract_Index', 'PRP_Name', 'Final_Affinity']).copy()

# ra_analysis_df = (
#     ra_analysis_df.groupby(['Abstract_Index', 'PRP_Name'], as_index=False)
#     .agg(
#         Final_Affinity=('Final_Affinity', 'max'),
#         # RA2025_IDs=('RA2025_ID', lambda x: ','.join(map(str, x.dropna().unique())))
#         )
# ) #TODO: insert in ra_analysis_df the RA2025_IDs that contributed to the max affinity for each abstract/PRP pair, and add as a column
# ra_analysis_df['Affinity_Level'] = ra_analysis_df['Final_Affinity'].apply(affinity_level_fixed)


# print('Prepared rows:', len(ra_analysis_df))
# display(ra_analysis_df.head(10))

In [ ]:

# ra_count_df = (
#     ra_analysis_df.groupby(['PRP_Name', 'Affinity_Level'], as_index=False)['Abstract_Index']
#     .nunique()
#     .rename(columns={'Abstract_Index': 'n_abstracts'})
# )

# fig = px.bar(
#     ra_count_df,
#     x='PRP_Name',
#     y='n_abstracts',
#     color='Affinity_Level',
#     barmode='group',
#     category_orders={'Affinity_Level': ['Low', 'Moderate', 'High']},
#     title='RA-to-PRP Abstract Counts by Affinity Level and PRP'
# )
# fig.update_layout(xaxis_title='PRP Name', yaxis_title='Number of Abstracts', legend_title='Affinity Level')
# fig.update_xaxes(tickangle=45)
# fig.show()

# display(ra_count_df.sort_values(['PRP_Name', 'Affinity_Level']))

# PRP Affinity Count

In [ ]:
# Build analysis table and error bars
prp_analysis_df = prp_adjudication_df.copy()
prp_analysis_df['Final_Affinity'] = pd.to_numeric(prp_analysis_df['Final_Affinity'], errors='coerce')
prp_analysis_df['Affinity_Level'] = prp_analysis_df['Final_Affinity'].apply(affinity_level_fixed)

In [ ]:
# # Deterministic counts
# prp_count_df = (
#     prp_analysis_df.groupby(['PRP_Name', 'Affinity_Level'], as_index=False)['Abstract_Index']
#     .nunique()
#     .rename(columns={'Abstract_Index': 'n_abstracts'})
# )

# fig = px.bar(
#     prp_count_df,
#     x='PRP_Name',
#     y='n_abstracts',
#     color='Affinity_Level',
#     barmode='group',
#     category_orders={'Affinity_Level': ['Low', 'Moderate', 'High']},
#     title='Deterministic Abstract Counts by Affinity Level and PRP'
# )
# fig.update_layout(xaxis_title='PRP Name', yaxis_title='Number of Abstracts', legend_title='Affinity Level')
# fig.update_xaxes(tickangle=45)
# fig.show()
# display(prp_count_df.sort_values(['PRP_Name', 'Affinity_Level']))